In [ ]:
from pyspark.sql import functions as F

# 1. Lecture de la couche Silver propre
df_silver = spark.table("silver_events")

# -------------------------------------------------------------
# DIMENSION 1 : dim_product
# -------------------------------------------------------------
dim_product = (
    df_silver
    .select("product_id", "category_id", "category_code", "brand")
    .dropDuplicates(["product_id"])
)
(dim_product.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dim_product")
)

# -------------------------------------------------------------
# DIMENSION 2 : dim_user
# -------------------------------------------------------------
dim_user = (
    df_silver
    .select("user_id")
    .dropDuplicates(["user_id"])
)
(dim_user.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dim_user")
)

# -------------------------------------------------------------
# DIMENSION 3 : dim_date (Générée dynamiquement)
# -------------------------------------------------------------
dim_date = (
    df_silver
    .select(F.to_date("event_time").alias("date"))
    .dropDuplicates(["date"])
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
)
(dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dim_date")
)

# -------------------------------------------------------------
# DIMENSION 4 : dim_time (Granularité horaire pour analyse fine)
# -------------------------------------------------------------
dim_time = (
    df_silver
    .select(F.hour("event_time").alias("hour"))
    .dropDuplicates(["hour"])
    .withColumn("time_bucket", 
        F.when(F.col("hour").between(6, 11), "Morning")
         .when(F.col("hour").between(12, 17), "Afternoon")
         .when(F.col("hour").between(18, 23), "Evening")
         .otherwise("Night")
    )
)
(dim_time.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dim_time")
)

# -------------------------------------------------------------
# TABLE DE FAITS : fact_events (Partitionnée par date)
# -------------------------------------------------------------
fact_events = (
    df_silver
    .select(
        "event_time",
        F.to_date("event_time").alias("date"),
        F.hour("event_time").alias("hour"),
        "event_type",
        "product_id",
        "user_id",
        "user_session",
        "price"
    )
)
(fact_events.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("date")
    .saveAsTable("fact_events")
)

print("✅ Couche GOLD terminée : Schéma en étoile complet créé avec succès !")

StatementMeta(, f9eec793-1295-4c7c-955d-06687c25ca80, 3, Finished, Available, Finished, False)

✅ Couche GOLD terminée : Schéma en étoile complet créé avec succès !
